## Semantic Caching lab
![flow](img/semantic-caching.gif)

Playground to try the [semantic caching policy](https://learn.microsoft.com/azure/api-management/azure-openai-semantic-cache-lookup-policy). 

The azure-openai-semantic-cache-lookup policy conducts a cache lookup of responses on Azure OpenAI Chat Completion API and Completion API requests from a pre-configured external cache. It operates by comparing the vector proximity of the prompt to prior requests and using a specific similarity score threshold. Caching responses helps reduce bandwidth and processing demands on the backend Azure OpenAI API, thus reducing latency perceived by API consumers.  

[View policy configuration](./terraform/modules/semantic_caching/semantic_caching.xml)

### ⚙️ Initialization

1. Get Az Cli authentication information
1. Get terraform outputs (prerequisite: run the [setup.ipynb](./setup.ipynb))

In [ ]:
import sys
sys.path.insert(1, './shared')  # add the shared directory to the Python path
import utils

terraform_dir = 'terraform'

output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run(f"terraform -chdir={terraform_dir} output -json", "Retrieved Terraform outputs", "Failed to retrieve Terraform outputs")

if output.success and output.json_data:
    apim_gateway_url = output.json_data['apim_gateway_url']['value']
    inference_api_path = output.json_data['semantic_caching']['value']['inference_api_path']
    inference_api_key = output.json_data['semantic_caching']['value']['inference_api_key']
    inference_api_version = output.json_data['semantic_caching']['value']['inference_api_version']
    rediscache_host = output.json_data['rediscache_host']['value']
    rediscache_key = output.json_data['rediscache_key']['value']
    rediscache_port = output.json_data['rediscache_port']['value']

    utils.print_info(f"APIM Gateway URL: {apim_gateway_url}")
    utils.print_info(f"Inference API path: {inference_api_path}")
    utils.print_info(f"Inference API key: {inference_api_key}")
    utils.print_info(f"Inference API version: {inference_api_version}")
    utils.print_info(f"Redis Cache Host: {rediscache_host}")
    utils.print_info(f"Redis Cache Key: {rediscache_key}")
    utils.print_info(f"Redis Cache Port: {rediscache_port}")

<a id='sdk'></a>
### 🧪 Make multiple calls using the Azure OpenAI Python SDK

The code below contains a list of questions that will be randomly selected and sent as prompts to the OpenAI API

In [ ]:
from openai import AzureOpenAI
import time, random

runs = 20
sleep_time_ms = 10
questions = ["How to Brew the Perfect Cup of Coffee?",
             "What are the steps to Craft the Ideal Espresso?",
             "Tell me how to create the best steaming Java?",
             "Explain how to make a caffeinated brewed beverage?"]
api_runs = []  # Response Times for each run
client = AzureOpenAI(
    azure_endpoint = f"{apim_gateway_url}/{inference_api_path}",
    api_key = inference_api_key,
    api_version = inference_api_version
)

for i in range(runs):
    print(f"▶️ Run {i+1}/{runs}:")
    random_question = random.choice(questions)
    print("💬 ", random_question)

    start_time = time.time()
    response = client.chat.completions.create(
        model = 'gpt-4.1-mini',
        messages = [
            {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
            {"role": "user", "content": random_question}
        ])
    response_time = time.time() - start_time

    print(f"⌚ {response_time:.2f} seconds")

    # Uncomment to print the response
    print(f"💬 {response.choices[0].message.content}\n")

    api_runs.append(response_time)
    time.sleep(sleep_time_ms/1000)

<a id='plot'></a>
### 🔍 Analyze Semantic Caching performance

The first request should take a longer time as it makes it all the way to the Azure OpenAI backend. The subsequent requests should be much quicker as they draw from the semantic cache. Note that making more than 20 requests may result in spikes similar to the first request. As we are using the cheapest, smallest Basic Redis cache (B0), the cache server will eventually return a 429, forcing API Management to make a longer request to the Azure OpenAI backend. This is expected as B0 is not intended for load scenarios.

In [ ]:
# plot the results
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['figure.figsize'] = [15, 5]
df = pd.DataFrame(api_runs, columns=['Response Time'])
df['Run'] = range(1, len(df) + 1)
df.plot(kind='bar', x='Run', y='Response Time', legend=False)
plt.title('Semantic Caching Performance')
plt.xlabel('Runs')
plt.ylabel('Response Time (s)')
plt.xticks(rotation=0)  # Set x-axis ticks to be the run numbers

average = df['Response Time'].mean()
plt.axhline(y=average, color='r', linestyle='--', label=f'Average: {average:.2f}')
plt.legend()

plt.show()

<a id='statistics'></a>
### 🔍 Show Redis Cache information

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import redis.asyncio as redis

async def get_redis_info():
    r = await redis.from_url(
        f"rediss://:{rediscache_key}@{rediscache_host}:{rediscache_port}"
    )

    info = await r.info()

    print("Redis Server Information:")
    print(f"Used Memory  : {info['used_memory_human']}")
    # Display the Redis info in a pandas DataFrame and plot it

    def convert_memory_to_bytes(memory_str):
        units = {"K": 1024, "M": 1024**2, "G": 1024**3}
        if memory_str[-1] in units:
            return float(memory_str[:-1]) * units[memory_str[-1]]
        return float(memory_str)

    redis_info = {
        'Metric': ['Cache Hits', 'Cache Misses', 'Evicted Keys', 'Expired Keys'],
        'Value': [info['keyspace_hits'], info['keyspace_misses'], info['evicted_keys'], info['expired_keys']]
    }

    df_redis_info = pd.DataFrame(redis_info)
    df_redis_info.plot(kind='barh', x='Metric', y='Value', legend=False)

    plt.title('Redis Server Information')
    plt.xlabel('Value')
    plt.ylabel('Metric')
    plt.show()

    await r.aclose()

await get_redis_info()

### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.